In [ ]:
# imports + load the 4 CSVs, fix converted column, check shapes
import pandas as pd, numpy as np, statsmodels.api as sm
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt

experiments = pd.read_csv("../data/experiments.csv")
assignments = pd.read_csv("../data/assignments.csv")
events = pd.read_csv("../data/events.csv")
users = pd.read_csv("../data/users.csv")

control = subset[subset.variant=="control"]["converted"].fillna(False).astype(int)
treat = subset[subset.variant=="treatment"]["converted"].fillna(False).astype(int)


print(experiments.shape, assignments.shape, events.shape, users.shape)

(10, 9) (93050, 5) (619048, 7) (46000, 8)


In [44]:
# keep only the 8 trustworthy experiments, then merge everything into one table
ANALYSABLE = ["EXP-001","EXP-002","EXP-003","EXP-004","EXP-006","EXP-007","EXP-008","EXP-009"]

assignments = assignments[assignments.experiment_id.isin(ANALYSABLE)]
events = events[events.experiment_id.isin(ANALYSABLE)]

merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")

print(merged.shape)
merged.head()

(476457, 17)


,assignment_id,experiment_id,user_id,variant,assigned_at,signup_date,country,device,acquisition_channel,pre_sessions_28d,pre_orders_28d,pre_revenue_28d,event_date,sessions,converted,orders,revenue_xaf
0,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-04,3.0,False,0.0,0.0
1,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-05,3.0,False,0.0,0.0
2,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-06,1.0,False,0.0,0.0
3,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-07,2.0,False,0.0,0.0
4,AS-0016202,EXP-002,UU736547,control,2025-11-08,2025-05-14,CF,ios,referral,4,4,3331,2025-11-10,2.0,False,0.0,0.0


In [45]:
# check for merge problems: missing device (bad merge to users) or missing converted (no matching event)
print(merged["device"].isna().sum())
print(merged["converted"].isna().sum())


0
117


In [46]:
print(len(merged))
print(476457 / len(merged))

476457
1.0


In [47]:
# compare actual values between the two tables to spot mismatches
print(assignments["user_id"].head(3).tolist())
print(events["user_id"].head(3).tolist())
print(assignments["experiment_id"].unique()[:3])
print(events["experiment_id"].unique()[:3])

['UU736547', 'UU730730', 'UU729627']
['UU714798', 'UU714798', 'UU714798']
['EXP-002' 'EXP-004' 'EXP-007']
['EXP-001' 'EXP-002' 'EXP-003']


In [48]:
# pick one real user from assignments and check if events has ANY row for that exact user+experiment
sample = assignments.iloc[0]
print(sample["user_id"], sample["experiment_id"])

match = events[(events.user_id == sample["user_id"]) & (events.experiment_id == sample["experiment_id"])]
print(len(match))


UU736547 EXP-002
10


In [49]:
# check for hidden whitespace or dtype mismatches between the join columns
print(repr(assignments["user_id"].iloc[0]))
print(repr(events["user_id"].iloc[0]))
print(assignments["user_id"].dtype, events["user_id"].dtype)
print(assignments["experiment_id"].dtype, events["experiment_id"].dtype)

'UU736547'
'UU714798'
object object
object object


In [50]:
# minimal test: merge assignments to events using ONLY that one known-good row
test_merge = assignments.iloc[[0]].merge(events, on=["experiment_id","user_id"], how="left")
print(len(test_merge))
print(test_merge[["user_id","experiment_id","event_date"]].head())

10
    user_id experiment_id  event_date
0  UU736547       EXP-002  2025-11-04
1  UU736547       EXP-002  2025-11-05
2  UU736547       EXP-002  2025-11-06
3  UU736547       EXP-002  2025-11-07
4  UU736547       EXP-002  2025-11-10


In [51]:
# check what's actually in your variables right now
print(len(assignments), len(events))
print(assignments.columns.tolist())
print(events.columns.tolist())

73618 475582
['assignment_id', 'experiment_id', 'user_id', 'variant', 'assigned_at']
['experiment_id', 'user_id', 'event_date', 'sessions', 'converted', 'orders', 'revenue_xaf']


In [52]:
# rerun the real merge fresh, and check 
merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")
print(merged["converted"].isna().sum(), len(merged))

117 476457


In [53]:
# do the merge in two separate visible steps instead of chaining
step1 = assignments.merge(users, on="user_id", how="left")
print(len(step1), step1["device"].isna().sum())

step2 = step1.merge(events, on=["experiment_id","user_id"], how="left")
print(len(step2), step2["converted"].isna().sum())


73618 0
476457 117


In [54]:
# check for any column name overlap between users and events besides the merge keys
print(set(users.columns) & set(events.columns))

{'user_id'}


In [55]:
# confirm step1's key columns still match assignments exactly
print(step1["user_id"].equals(assignments["user_id"].reset_index(drop=True)))
print(step1["experiment_id"].equals(assignments["experiment_id"].reset_index(drop=True)))
print(step1[["user_id","experiment_id"]].dtypes)
print(events[["user_id","experiment_id"]].dtypes)
sample = step1.iloc[0]
match = events[(events.user_id == sample["user_id"]) & (events.experiment_id == sample["experiment_id"])]
print(sample["user_id"], sample["experiment_id"], len(match))

True
True
user_id          object
experiment_id    object
dtype: object
user_id          object
experiment_id    object
dtype: object
UU736547 EXP-002 10


In [56]:
# reset indexes explicitly, then merge fresh and check right away
a = assignments.reset_index(drop=True)
e = events.reset_index(drop=True)
test = a.merge(e, on=["experiment_id","user_id"], how="inner")
print(len(test))


476340


In [57]:
# clean, confirmed-working merge
merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")
print(len(merged), merged["converted"].isna().sum())
print(events["converted"].head(10))
print(events["converted"].unique())

476457 117
0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: converted, dtype: bool
[False  True]


In [58]:
# check what converted actually looked like right after loading, before the broken .map() line
events2 = pd.read_csv("../data/events.csv")
print(events2["converted"].dtype)
print(events2["converted"].unique())

bool
[False  True]


In [59]:
# reload all 4 files clean (converted stays boolean automatically), then rebuild the merge
experiments = pd.read_csv("../data/experiments.csv")
assignments = pd.read_csv("../data/assignments.csv")
events = pd.read_csv("../data/events.csv")
users = pd.read_csv("../data/users.csv")

ANALYSABLE = ["EXP-001","EXP-002","EXP-003","EXP-004","EXP-006","EXP-007","EXP-008","EXP-009"]
assignments = assignments[assignments.experiment_id.isin(ANALYSABLE)]
events = events[events.experiment_id.isin(ANALYSABLE)]

merged = assignments.merge(users, on="user_id", how="left").merge(events, on=["experiment_id","user_id"], how="left")
print(len(merged), merged["converted"].isna().sum(), merged["device"].isna().sum())

476457 117 0


In [60]:
# isolate one experiment to build and test the regression code on
one_test = merged[merged.experiment_id == "EXP-001"].copy()
print(one_test.shape)
print(one_test["variant"].value_counts())

(57071, 17)
variant
control      28725
treatment    28346
Name: count, dtype: int64


In [61]:
# convert categories to 0/1 columns, build variant as 0/1, run regression
data_dummies = pd.get_dummies(one_test, columns=["device","country","acquisition_channel"], drop_first=True)
data_dummies["variant_treated"] = (data_dummies["variant"] == "treatment").astype(int)

covariate_cols = [c for c in data_dummies.columns if c.startswith(("device_","country_","acquisition_channel_"))]
X = sm.add_constant(data_dummies[["variant_treated"] + covariate_cols])
y = data_dummies["converted"].fillna(False).astype(int)

model = sm.OLS(y, X).fit()
print(model.summary())

C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1011300387.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = data_dummies["converted"].fillna(False).astype(int)


                            OLS Regression Results                            
Dep. Variable:              converted   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.268
Date:                Fri, 04 Sep 2026   Prob (F-statistic):              0.213
Time:                        03:04:53   Log-Likelihood:                -15238.
No. Observations:               57071   AIC:                         3.051e+04
Df Residuals:                   57055   BIC:                         3.065e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

In [62]:
# pull out just the effect and CI for variant_treated, save as a row
effect = model.params["variant_treated"]
ci_low, ci_high = model.conf_int().loc["variant_treated"]
result_row = {"experiment_id": "EXP-001", "effect": round(effect,4), "ci_low": round(ci_low,4), "ci_high": round(ci_high,4)}
print(result_row)

{'experiment_id': 'EXP-001', 'effect': np.float64(0.0008), 'ci_low': -0.0044, 'ci_high': 0.0059}


In [63]:
# collapse to one row per user (sum revenue across their events), then apply CUPED
user_level = one_test.groupby("user_id").agg(
    variant=("variant","first"),
    revenue=("revenue_xaf","sum"),
    pre_revenue_28d=("pre_revenue_28d","first")
).reset_index()

covariance = np.cov(user_level["revenue"], user_level["pre_revenue_28d"])[0][1]
variance = np.var(user_level["pre_revenue_28d"], ddof=1)
theta = covariance / variance

mean_pre = user_level["pre_revenue_28d"].mean()
user_level["adjusted_revenue"] = user_level["revenue"] - theta * (user_level["pre_revenue_28d"] - mean_pre)

print("theta:", theta)
user_level.head()

theta: -0.007710279451147819


,user_id,variant,revenue,pre_revenue_28d,adjusted_revenue
0,UU700001,treatment,15528.0,29997,15645.235031
1,UU700004,treatment,0.0,31231,126.749516
2,UU700012,control,40060.0,24282,40133.170784
3,UU700013,treatment,0.0,13655,-8.766356
4,UU700017,treatment,14776.0,21386,14826.841815


In [64]:
# compare CI width using raw revenue vs CUPED-adjusted revenue
def ci_width(a, b):
    diff = b.mean() - a.mean()
    se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    return 2 * 1.96 * se

control_raw = user_level[user_level.variant=="control"]["revenue"]
treat_raw = user_level[user_level.variant=="treatment"]["revenue"]
control_adj = user_level[user_level.variant=="control"]["adjusted_revenue"]
treat_adj = user_level[user_level.variant=="treatment"]["adjusted_revenue"]

width_before = ci_width(control_raw, treat_raw)
width_after = ci_width(control_adj, treat_adj)
print(f"before: {width_before:.2f}, after: {width_after:.2f}, reduction: {(1-width_after/width_before)*100:.1f}%")

before: 779.41, after: 779.38, reduction: 0.0%


In [65]:
# check how strongly pre-period revenue actually predicts during-experiment revenue
print(user_level[["revenue","pre_revenue_28d"]].corr())

                  revenue  pre_revenue_28d
revenue          1.000000        -0.008845
pre_revenue_28d -0.008845         1.000000


In [66]:
# save CUPED result for this experiment
cuped_result = {"experiment_id": "EXP-001", "width_before": round(779.41,2), "width_after": round(779.38,2), "reduction_pct": 0.0}
print(cuped_result)

{'experiment_id': 'EXP-001', 'width_before': 779.41, 'width_after': 779.38, 'reduction_pct': 0.0}


In [70]:
# check effect by device, country, and channel separately - hunting for a reversal
segment_rows = []
for col in ["device","country","acquisition_channel"]:
    for val in one_test[col].dropna().unique():
        subset = one_test[one_test[col]==val]
        control = subset[subset.variant=="control"]["converted"].fillna(False).astype(int)
        treat = subset[subset.variant=="treatment"]["converted"].fillna(False).astype(int)
        if len(control)<5 or len(treat)<5: continue
        diff = treat.mean()-control.mean()
        se = np.sqrt(control.var(ddof=1)/len(control)+treat.var(ddof=1)/len(treat))
        segment_rows.append({"segment_type":col,"segment_value":val,"effect":round(diff,4),"ci_low":round(diff-1.96*se,4),"ci_high":round(diff+1.96*se,4)})

segment_df = pd.DataFrame(segment_rows)
print(segment_df.sort_values("effect"))

C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1712299424.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  control = subset[subset.variant=="control"]["converted"].fillna(False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1712299424.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  treat = subset[subset.variant=="treatment"]["converted"].fillna(False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1712299424.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and 

           segment_type segment_value  effect  ci_low  ci_high
7               country            CI -0.0167 -0.0350   0.0016
11              country            GH -0.0140 -0.0321   0.0042
15  acquisition_channel       partner -0.0083 -0.0211   0.0045
5               country            GA -0.0082 -0.0265   0.0100
6               country            CF -0.0071 -0.0240   0.0099
1                device           ios -0.0032 -0.0159   0.0096
14  acquisition_channel      referral -0.0030 -0.0163   0.0102
13  acquisition_channel   paid search -0.0030 -0.0158   0.0099
0                device       android  0.0011 -0.0052   0.0074
16  acquisition_channel   paid social  0.0013 -0.0112   0.0137
10              country            NG  0.0015 -0.0173   0.0204
2                device           web  0.0035 -0.0093   0.0163
9               country            TD  0.0058 -0.0125   0.0240
8               country            SN  0.0072 -0.0101   0.0246
4               country            CM  0.0073 -0.0017  

C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1712299424.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  control = subset[subset.variant=="control"]["converted"].fillna(False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1712299424.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  treat = subset[subset.variant=="treatment"]["converted"].fillna(False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\1712299424.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and 

In [73]:
# check effect by device, country, and channel separately - hunting for a reversal
segment_rows = []
for col in ["device","country","acquisition_channel"]:
    for val in one_test[col].dropna().unique():
        subset = one_test[one_test[col]==val]
        control = subset[subset.variant=="control"]["converted"].fillna(False).infer_objects(copy=False).astype(int)
        treat = subset[subset.variant=="treatment"]["converted"].fillna(False).infer_objects(copy=False).astype(int)
        if len(control)<5 or len(treat)<5: continue
        diff = treat.mean()-control.mean()
        se = np.sqrt(control.var(ddof=1)/len(control)+treat.var(ddof=1)/len(treat))
        segment_rows.append({"segment_type":col,"segment_value":val,"effect":round(diff,4),"ci_low":round(diff-1.96*se,4),"ci_high":round(diff+1.96*se,4)})

segment_df = pd.DataFrame(segment_rows)
print(segment_df.sort_values("effect"))

C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\3598915221.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  control = subset[subset.variant=="control"]["converted"].fillna(False).infer_objects(copy=False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\3598915221.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  treat = subset[subset.variant=="treatment"]["converted"].fillna(False).infer_objects(copy=False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\3598915221.py:6: FutureWarning: Downcasting object dtype 

           segment_type segment_value  effect  ci_low  ci_high
7               country            CI -0.0167 -0.0350   0.0016
11              country            GH -0.0140 -0.0321   0.0042
15  acquisition_channel       partner -0.0083 -0.0211   0.0045
5               country            GA -0.0082 -0.0265   0.0100
6               country            CF -0.0071 -0.0240   0.0099
1                device           ios -0.0032 -0.0159   0.0096
14  acquisition_channel      referral -0.0030 -0.0163   0.0102
13  acquisition_channel   paid search -0.0030 -0.0158   0.0099
0                device       android  0.0011 -0.0052   0.0074
16  acquisition_channel   paid social  0.0013 -0.0112   0.0137
10              country            NG  0.0015 -0.0173   0.0204
2                device           web  0.0035 -0.0093   0.0163
9               country            TD  0.0058 -0.0125   0.0240
8               country            SN  0.0072 -0.0101   0.0246
4               country            CM  0.0073 -0.0017  

C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\3598915221.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  control = subset[subset.variant=="control"]["converted"].fillna(False).infer_objects(copy=False).astype(int)
C:\Users\Endeavor\AppData\Local\Temp\ipykernel_16836\3598915221.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  treat = subset[subset.variant=="treatment"]["converted"].fillna(False).infer_objects(copy=False).astype(int)
